=============================================================================
RESPIRATORY DISEASE CLASSIFIER — LAYER 2: CNN + FULL PIPELINE COMBINER
=============================================================================
PURPOSE:
  Part A — Layer 2 CNN:
      Multi-class classification: Asthma vs COPD vs COVID-19
      Input: Raw WAV files (cough + vowel per patient)
      Architecture: Two-stream custom CNN on Mel Spectrograms
                    (one branch for cough, one for vowel, merged)

  Part B — Combined Pipeline:
      Stacks Layer 1 (HistGradBoost or XGBoost) + Layer 2 (CNN)
      into a single TwoLayerClassifier object
      Saves and reloads the full pipeline

USAGE (Google Colab with SSD-mounted data):
  - Mount your SSD
  - Update paths in CONFIG
  - Run sections in order: A then B

METRICS: Accuracy, Precision, Recall (macro+per-class), F1, Confusion Matrix
=============================================================================

=============================================================================
SECTION 0: INSTALL & IMPORTS
=============================================================================

In [15]:
import os                            # File and path operations
import time                          # Training duration tracking
import random                        # Random seed for reproducibility
import pickle                        # Saving the combined pipeline object
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch                         # PyTorch core
import torch.nn as nn                # Neural network layers
import torch.optim as optim          # Optimisers
from torch.utils.data import Dataset, DataLoader   # Data pipeline
import torchaudio                    # Audio loading and transforms
import torchaudio.transforms as T    # MelSpectrogram, AmplitudeToDB, etc.
import librosa                       # Audio loading fallback and utilities
import joblib                        # Loading Layer 1 sklearn model
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

=============================================================================
SECTION 1: REPRODUCIBILITY SEED
=============================================================================

In [16]:
def set_seed(seed=42):
    """
    Sets random seeds across all libraries for reproducible results.
    Must be called before any model initialisation or data loading.

    Args:
        seed (int): The seed value to use everywhere.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Make CuDNN deterministic (slightly slower, but reproducible)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"[SEED] All random seeds set to {seed}.")

In [17]:
set_seed(42)

[SEED] All random seeds set to 42.


=============================================================================
SECTION 2: CONFIGURATION
=============================================================================

In [39]:
CONFIG = {
    # ---- PATHS ----
    # CSV that maps patient_id → cough_wav_path, vowel_wav_path, label
    # This is the custom dataset you said you'll build independently.
    # Expected columns: ['id', 'cough_path', 'vowel_path', 'label']
    # Labels: 'asthma', 'copd', 'covid' (healthy patients excluded)
    "train_csv"  : r"C:\Users\Admin\OneDrive\Desktop\. respiratory\1 final datasets, data info file, data distributions, final models\CNN models\only_paths_train_vowel_and_cough_ros_healthy_dropped.csv",
    "val_csv"    : r"C:\Users\Admin\OneDrive\Desktop\. respiratory\1 final datasets, data info file, data distributions, final models\CNN models\only_paths_val_vowel_and_cough_healthy_dropped.csv",
    "test_csv"   : r"C:\Users\Admin\OneDrive\Desktop\. respiratory\1 final datasets, data info file, data distributions, final models\CNN models\only_paths_test_vowel_and_cough_healthy_dropped.csv",

    # Where to save the CNN model and combined pipeline
    "save_dir": r"C:\Users\Admin\OneDrive\Desktop\layer2_cnn.joblib",
    "cnn_filename": "layer2_cnn.pth",          # CNN weights file
    "pipeline_filename": "full_pipeline.pkl",        # Combined pipeline

    # Path to your saved Layer 1 model (choose ONE of the two):
    # XGBoost (comment out A and uncomment B if using XGBoost):
    "layer1_path": r"C:\Users\Admin\OneDrive\Desktop\layer1_xgboost.joblib",
    "layer1_type": "xgboost",

    # ---- AUDIO PREPROCESSING ----
    "sample_rate"   : 16000,     # Resample all audio to 16 kHz
    "clip_duration" : 5.0,       # Fixed clip length in seconds (pad/trim to this)
    "n_mels"        : 128,       # Number of mel frequency bins
    "n_fft"         : 1024,      # FFT window size (controls frequency resolution)
    "hop_length"    : 512,       # Hop between FFT windows (controls time resolution)
    "f_min"         : 20,        # Minimum frequency for mel filterbank (Hz)
    "f_max"         : 8000,      # Maximum frequency (= sample_rate / 2 for 16kHz)

    # ---- TRAINING ----
    "batch_size"    : 32,        # Samples per batch
    "num_epochs"    : 100,       # Maximum training epochs
    "learning_rate" : 1e-4,      # AdamW initial learning rate
    "weight_decay"  : 1e-4,      # AdamW L2 regularisation
    "early_stop_patience": 10,   # Stop if val F1 doesn't improve for N epochs
    "label_smoothing": 0.1,      # Label smoothing epsilon for CrossEntropyLoss
    "num_workers"   : 2,         # DataLoader worker processes (set to 0 if errors)
    "num_classes"   : 3,         # asthma, copd, covid

    # ---- MISC ----
    "save_plots"    : True,      # Save confusion matrix plots to disk
}

In [19]:
# Class label mapping for Layer 2
# The CNN outputs 3 logits → map to disease names
LABEL_MAP = {
    "asthma": 1,
    "copd"  : 2,
    "covid" : 3,
}
# Reverse mapping (index → name) for reporting
IDX_TO_LABEL = {v: k for k, v in LABEL_MAP.items()}

In [20]:
# Create save directory
os.makedirs(CONFIG["save_dir"], exist_ok=True)

In [21]:
# Determine device: use GPU if available, else CPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[DEVICE] Using: {DEVICE}")

[DEVICE] Using: cpu


=============================================================================
SECTION 3: MEL SPECTROGRAM TRANSFORM
=============================================================================

In [22]:
def build_mel_transform(sample_rate, n_mels, n_fft, hop_length, f_min, f_max):
    """
    Builds a torchaudio transform pipeline that converts a raw waveform tensor
    into a log-scaled Mel spectrogram.

    Pipeline:
        Waveform → MelSpectrogram → AmplitudeToDB (log scale)

    Why log scale?
        Human hearing is logarithmic. Log-Mel spectrograms compress the dynamic
        range and make the CNN learn perceptually meaningful patterns.

    Args:
        sample_rate (int): Audio sample rate after resampling.
        n_mels      (int): Number of mel frequency bins (height of spectrogram).
        n_fft       (int): FFT window size in samples.
        hop_length  (int): Hop between consecutive FFT windows.
        f_min       (float): Minimum frequency for mel filterbank.
        f_max       (float): Maximum frequency for mel filterbank.

    Returns:
        transform (nn.Sequential): Composed torchaudio transform.
    """

    mel_transform = T.MelSpectrogram(
        sample_rate=sample_rate,
        n_fft=n_fft,
        hop_length=hop_length,
        n_mels=n_mels,
        f_min=f_min,
        f_max=f_max,
        power=2.0,         # Power spectrogram (magnitude squared)
    )

    # Convert power spectrogram to dB scale (log)
    # top_db=80: clamps the dynamic range to 80 dB (removes very quiet noise)
    db_transform = T.AmplitudeToDB(stype="power", top_db=80)

    # Chain them together
    transform = nn.Sequential(mel_transform, db_transform)
    return transform

In [23]:
# Build the shared transform (used by the Dataset class)
MEL_TRANSFORM = build_mel_transform(
    sample_rate=CONFIG["sample_rate"],
    n_mels=CONFIG["n_mels"],
    n_fft=CONFIG["n_fft"],
    hop_length=CONFIG["hop_length"],
    f_min=CONFIG["f_min"],
    f_max=CONFIG["f_max"],
)

=============================================================================
SECTION 4: AUDIO LOADING UTILITY
=============================================================================

In [24]:
def load_and_preprocess_audio(wav_path, target_sr, clip_duration, transform):
    """
    Loads a WAV file, resamples it, pads or trims to a fixed duration,
    and converts it to a log-Mel spectrogram tensor.

    Steps:
        1. Load WAV with torchaudio (fast, handles most formats)
        2. Convert stereo to mono by averaging channels
        3. Resample to target_sr
        4. Pad with zeros if shorter than clip_duration, trim if longer
        5. Apply MelSpectrogram + AmplitudeToDB
        6. Normalise spectrogram to zero mean, unit variance

    Args:
        wav_path      (str)        : Path to the .wav file.
        target_sr     (int)        : Target sample rate (e.g., 16000).
        clip_duration (float)      : Fixed duration in seconds to pad/trim to.
        transform     (nn.Sequential): Mel spectrogram transform pipeline.

    Returns:
        spec (torch.Tensor): Log-Mel spectrogram of shape (1, n_mels, time_frames).
                             Shape (1, 128, ~157) for 5s clips at hop_length=512.
    """

    # --- Load audio ---
    try:
        waveform, sr = torchaudio.load(wav_path)
        # waveform shape: (channels, samples)
    except Exception:
        # Fallback to librosa if torchaudio fails (some compressed formats)
        y, sr = librosa.load(wav_path, sr=None, mono=True)
        waveform = torch.tensor(y).unsqueeze(0)   # shape: (1, samples)

    # --- Convert stereo to mono (average channels) ---
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    # waveform shape is now (1, samples)

    # --- Resample to target sample rate ---
    if sr != target_sr:
        resampler = T.Resample(orig_freq=sr, new_freq=target_sr)
        waveform  = resampler(waveform)

    # --- Pad or trim to fixed number of samples ---
    target_samples = int(target_sr * clip_duration)
    current_samples = waveform.shape[1]

    if current_samples < target_samples:
        # Pad with zeros at the end
        pad_amount = target_samples - current_samples
        waveform = torch.nn.functional.pad(waveform, (0, pad_amount))
    else:
        # Trim from the beginning (or centre — trimming from start is simpler)
        waveform = waveform[:, :target_samples]

    # --- Convert to log-Mel spectrogram ---
    # spec shape: (1, n_mels, time_frames)
    spec = transform(waveform)

    # --- Per-sample normalisation (zero mean, unit variance) ---
    # Normalise each spectrogram independently to handle volume differences
    mean = spec.mean()
    std  = spec.std() + 1e-8    # Add small epsilon to prevent division by zero
    spec = (spec - mean) / std

    return spec   # shape: (1, n_mels, time_frames)

=============================================================================
SECTION 5: SPECAUGMENT — DATA AUGMENTATION FOR SPECTROGRAMS
=============================================================================

In [25]:
def apply_specaugment(spec, freq_mask_param=20, time_mask_param=40):
    """
    Applies SpecAugment to a spectrogram during training.

    SpecAugment randomly masks:
        - Frequency bands: zero out F consecutive mel bins
        - Time steps:      zero out T consecutive time frames

    Why it works:
        Forces the CNN to learn from partial information, acting as a strong
        regulariser. Especially helpful for small audio datasets.

    Args:
        spec              (torch.Tensor): Spectrogram of shape (1, n_mels, time).
        freq_mask_param   (int): Max frequency bands to mask (F parameter).
        time_mask_param   (int): Max time steps to mask (T parameter).

    Returns:
        Augmented spectrogram of same shape.
    """

    # Frequency masking: randomly zeros out up to freq_mask_param mel bins
    freq_masker = T.FrequencyMasking(freq_mask_param=freq_mask_param)
    spec = freq_masker(spec)

    # Time masking: randomly zeros out up to time_mask_param time frames
    time_masker = T.TimeMasking(time_mask_param=time_mask_param)
    spec = time_masker(spec)

    return spec

=============================================================================
SECTION 6: PYTORCH DATASET CLASS
=============================================================================

In [44]:
class RespiratoryAudioDataset(Dataset):
    """
    Custom PyTorch Dataset for the two-stream (cough + vowel) audio classifier.

    Each sample contains:
        - cough WAV path
        - vowel WAV path
        - integer disease label

    Label mapping:
        0 = asthma
        1 = copd
        2 = covid
    """

    def __init__(
        self,
        csv_path,
        transform,
        augment=False,
        sample_rate=16000,
        clip_duration=5.0,
    ):
        super().__init__()

        # Load CSV
        # Required columns:
        # ['id', 'cough_path', 'vowel_path', 'disease']
        self.df = pd.read_csv(csv_path)

        self.transform = transform
        self.augment = augment
        self.sample_rate = sample_rate
        self.clip_duration = clip_duration

        # Labels already integer encoded
        self.labels = self.df["disease"].astype(int)

        # Safety check
        valid_labels = {0, 1, 2}

        invalid = self.labels[~self.labels.isin(valid_labels)]

        if len(invalid) > 0:
            raise ValueError(
                f"Invalid labels found in {csv_path}: {invalid.unique()}"
            )

        print(f"[DATASET] Loaded {len(self.df)} samples from {csv_path}")

        class_names = {
            0: "Asthma",
            1: "COPD",
            2: "COVID",
        }

        # Report class distribution
        for idx, name in class_names.items():
            count = (self.labels == idx).sum()
            print(f"  {name}: {count}")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        # --- Load cough spectrogram ---
        cough_spec = load_and_preprocess_audio(
            wav_path=row["cough_path"],
            target_sr=self.sample_rate,
            clip_duration=self.clip_duration,
            transform=self.transform,
        )

        # --- Load vowel spectrogram ---
        vowel_spec = load_and_preprocess_audio(
            wav_path=row["vowel_path"],
            target_sr=self.sample_rate,
            clip_duration=self.clip_duration,
            transform=self.transform,
        )

        # Concatenate along time dimension
        combined_spec = torch.cat([cough_spec, vowel_spec], dim=2)

        # Apply SpecAugment during training
        if self.augment:
            combined_spec = apply_specaugment(
                combined_spec,
                freq_mask_param=20,
                time_mask_param=40,
            )

        # Label tensor
        label = torch.tensor(self.labels.iloc[idx], dtype=torch.long)

        return combined_spec, label

=============================================================================
SECTION 7: DATALOADER CREATION
=============================================================================

In [42]:
def create_dataloaders(config, transform):
    """
    Creates train, validation, and test DataLoaders.

    Training DataLoader uses SpecAugment; val and test do not.

    Args:
        config    (dict): The CONFIG dictionary.
        transform : The mel spectrogram transform.

    Returns:
        train_loader, val_loader, test_loader (DataLoader objects)
    """

    # --- Datasets ---
    train_dataset = RespiratoryAudioDataset(
        csv_path=config["train_csv"],
        transform=transform,
        augment=True,                          # SpecAugment ON for training
        sample_rate=config["sample_rate"],
        clip_duration=config["clip_duration"],
    )

    val_dataset = RespiratoryAudioDataset(
        csv_path=config["val_csv"],
        transform=transform,
        augment=False,                         # No augmentation for validation
        sample_rate=config["sample_rate"],
        clip_duration=config["clip_duration"],
    )

    test_dataset = RespiratoryAudioDataset(
        csv_path=config["test_csv"],
        transform=transform,
        augment=False,                         # No augmentation for test
        sample_rate=config["sample_rate"],
        clip_duration=config["clip_duration"],
    )

    # --- DataLoaders ---
    train_loader = DataLoader(
        train_dataset,
        batch_size=config["batch_size"],
        shuffle=True,                          # Shuffle training data each epoch
        num_workers=config["num_workers"],
        pin_memory=True,                       # Faster CPU→GPU transfer
        drop_last=True,                        # Drop last incomplete batch
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=config["batch_size"],
        shuffle=False,                         # No shuffle for val/test
        num_workers=config["num_workers"],
        pin_memory=True,
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=config["batch_size"],
        shuffle=False,
        num_workers=config["num_workers"],
        pin_memory=True,
    )

    print(f"\n[LOADER] Train batches: {len(train_loader)}")
    print(f"[LOADER] Val batches  : {len(val_loader)}")
    print(f"[LOADER] Test batches : {len(test_loader)}")

    return train_loader, val_loader, test_loader

=============================================================================
SECTION 8: CNN ARCHITECTURE — ConvBlock and RespiratoryNet
=============================================================================

In [28]:
class ConvBlock(nn.Module):
    """
    A single convolutional block: Conv2D → BatchNorm → ReLU → MaxPool.

    Each block progressively:
        - Extracts local patterns (Conv2D)
        - Normalises activations for training stability (BatchNorm)
        - Applies non-linearity (ReLU)
        - Reduces spatial dimensions (MaxPool)

    Args:
        in_channels  (int): Number of input channels.
        out_channels (int): Number of output channels (filters to learn).
        kernel_size  (int): Convolution kernel size (default 3×3).
        pool_size    (int): MaxPool window size (default 2×2).
        dropout_p    (float): Dropout probability after pooling (0 = disabled).
    """

    def __init__(self, in_channels, out_channels, kernel_size=3, pool_size=2, dropout_p=0.0):
        super().__init__()

        self.block = nn.Sequential(
            # Convolution: learns spatial patterns (local frequency-time features)
            nn.Conv2d(
                in_channels, out_channels,
                kernel_size=kernel_size,
                padding=kernel_size // 2,    # 'same' padding: preserves spatial size
                bias=False,                  # Bias is redundant when using BatchNorm
            ),
            # Batch Normalisation: normalises each channel across the batch
            # Speeds up training, allows higher learning rates, regularises
            nn.BatchNorm2d(out_channels),

            # ReLU: introduces non-linearity; zeros out negative activations
            nn.ReLU(inplace=True),

            # MaxPool: halves both height and width
            # Reduces spatial resolution, increases receptive field
            nn.MaxPool2d(kernel_size=pool_size, stride=pool_size),

            # Dropout: randomly zeros activations during training (regularisation)
            nn.Dropout2d(p=dropout_p),
        )

    def forward(self, x):
        """Forward pass through the conv block."""
        return self.block(x)

In [29]:
class RespiratoryNet(nn.Module):
    """
    Custom 4-block CNN for respiratory disease classification from spectrograms.

    Architecture:
        Input: (batch, 1, n_mels, 2*time_frames)
               e.g., (32, 1, 128, 314) for 5-second clips at hop=512

        Block 1: Conv(1→32)   + BN + ReLU + MaxPool(2) + Dropout(0.2)
        Block 2: Conv(32→64)  + BN + ReLU + MaxPool(2) + Dropout(0.3)
        Block 3: Conv(64→128) + BN + ReLU + MaxPool(2) + Dropout(0.3)
        Block 4: Conv(128→256)+ BN + ReLU + MaxPool(2) + Dropout(0.4)

        GlobalAveragePooling2D  → (batch, 256)
        Dense(256 → 128) + BN + ReLU + Dropout(0.4)
        Dense(128 → num_classes)   # Raw logits (softmax applied by loss function)

    Design notes:
        - Filter doubling (32→64→128→256): standard pattern for learning
          increasingly abstract features
        - Global Average Pooling instead of Flatten: fewer parameters,
          better generalisation, agnostic to time dimension length
        - No softmax on output: nn.CrossEntropyLoss expects raw logits

    Args:
        num_classes  (int): Number of output classes (3 for asthma/copd/covid).
        n_mels       (int): Height of input spectrogram (number of mel bins).
    """

    def __init__(self, num_classes=3, n_mels=128):
        super().__init__()

        # --- 4 Convolutional Blocks ---
        # Each block doubles channels, halves spatial dimensions
        self.conv_blocks = nn.Sequential(
            ConvBlock(in_channels=1,   out_channels=32,  dropout_p=0.2),   # Block 1
            ConvBlock(in_channels=32,  out_channels=64,  dropout_p=0.3),   # Block 2
            ConvBlock(in_channels=64,  out_channels=128, dropout_p=0.3),   # Block 3
            ConvBlock(in_channels=128, out_channels=256, dropout_p=0.4),   # Block 4
        )

        # --- Global Average Pooling ---
        # Collapses (batch, 256, H, W) → (batch, 256)
        # Takes the average across all spatial positions per channel
        # Much better than Flatten: handles variable input lengths, fewer params
        self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))

        # --- Fully Connected Classifier Head ---
        self.classifier = nn.Sequential(
            nn.Linear(256, 128),       # Dense: 256 → 128
            nn.BatchNorm1d(128),       # Normalise hidden activations
            nn.ReLU(inplace=True),     # Non-linearity
            nn.Dropout(p=0.4),         # 40% dropout for regularisation
            nn.Linear(128, num_classes),  # Output: raw logits for each class
        )

    def forward(self, x):
        """
        Forward pass through the full network.

        Args:
            x (torch.Tensor): Input spectrogram (batch, 1, n_mels, time).

        Returns:
            logits (torch.Tensor): Raw class scores (batch, num_classes).
        """

        # Pass through conv blocks
        x = self.conv_blocks(x)         # → (batch, 256, H', W')

        # Global average pooling
        x = self.global_avg_pool(x)     # → (batch, 256, 1, 1)

        # Flatten to 1D per sample
        x = x.view(x.size(0), -1)       # → (batch, 256)

        # Classifier head → raw logits
        logits = self.classifier(x)     # → (batch, num_classes)

        return logits

    def count_parameters(self):
        """Returns the total number of trainable parameters."""
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

=============================================================================
SECTION 9: TRAINING UTILITIES
=============================================================================

In [49]:
def compute_class_weights(train_csv, device):
    """
    Computes inverse-frequency class weights for multiclass training.

    Labels:
        0 = asthma
        1 = copd
        2 = covid
    """

    df = pd.read_csv(train_csv)

    # Integer labels directly from CSV
    labels = df["disease"].astype(int)

    num_classes = 3

    # Count samples per class
    counts = np.array([
        (labels == i).sum()
        for i in range(num_classes)
    ], dtype=np.float32)

    print(f"\n[CLASS WEIGHTS] Class counts: {counts}")

    # Inverse-frequency weighting
    weights = counts.sum() / (num_classes * counts)

    # Normalize weights
    weights = weights / weights.sum() * num_classes

    weights = torch.tensor(weights, dtype=torch.float32).to(device)

    print(f"[CLASS WEIGHTS] Weights: {weights}")

    return weights

In [31]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    """
    Runs one training epoch: forward pass → loss → backprop → update.

    Args:
        model     : RespiratoryNet (or any nn.Module).
        loader    : Training DataLoader.
        optimizer : AdamW optimizer.
        criterion : Loss function (CrossEntropyLoss with label smoothing).
        device    : torch.device (cuda or cpu).

    Returns:
        avg_loss (float): Mean training loss over all batches.
        accuracy (float): Training accuracy over all batches.
    """

    model.train()   # Set model to training mode (enables Dropout, BatchNorm tracking)

    total_loss = 0.0
    all_preds  = []
    all_labels = []

    for batch_idx, (specs, labels) in enumerate(loader):
        # Move data to GPU/CPU
        specs  = specs.to(device)
        labels = labels.to(device)

        # --- Zero gradients from previous batch ---
        optimizer.zero_grad()

        # --- Forward pass ---
        logits = model(specs)            # → (batch, num_classes)

        # --- Compute loss ---
        loss = criterion(logits, labels)

        # --- Backward pass: compute gradients ---
        loss.backward()

        # --- Gradient clipping: prevents exploding gradients ---
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        # --- Update model weights ---
        optimizer.step()

        # Accumulate loss and predictions
        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    accuracy = accuracy_score(all_labels, all_preds)
    return avg_loss, accuracy

In [32]:
@torch.no_grad()   # Disable gradient computation for validation (saves memory)
def validate_one_epoch(model, loader, criterion, device):
    """
    Evaluates the model on a validation or test DataLoader.

    @torch.no_grad() decorator disables gradient tracking —
    not needed during evaluation, saves memory and speeds up inference.

    Args:
        model     : RespiratoryNet.
        loader    : Validation or test DataLoader.
        criterion : Loss function.
        device    : torch.device.

    Returns:
        avg_loss (float): Mean validation loss.
        accuracy (float): Validation accuracy.
        f1_macro (float): Macro-averaged F1 score (PRIMARY metric).
        all_preds  (list): All predicted class indices.
        all_labels (list): All true class indices.
    """

    model.eval()   # Evaluation mode: disables Dropout, freezes BatchNorm stats

    total_loss = 0.0
    all_preds  = []
    all_labels = []

    for specs, labels in loader:
        specs  = specs.to(device)
        labels = labels.to(device)

        logits = model(specs)
        loss   = criterion(logits, labels)

        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    accuracy = accuracy_score(all_labels, all_preds)
    f1_macro = f1_score(all_labels, all_preds, average="macro", zero_division=0)

    return avg_loss, accuracy, f1_macro, all_preds, all_labels

=============================================================================
SECTION 10: TRAINING LOOP WITH EARLY STOPPING
=============================================================================

In [51]:
def train_cnn(model, train_loader, val_loader, config, device):
    """
    Full training loop with:
        - AdamW optimiser
        - ReduceLROnPlateau scheduler (halves LR when val F1 plateaus)
        - Label smoothing in CrossEntropyLoss
        - Inverse-frequency class weighting
        - Early stopping on validation macro F1
        - Best model checkpoint saving

    Args:
        model        : RespiratoryNet instance.
        train_loader : Training DataLoader.
        val_loader   : Validation DataLoader.
        config       : CONFIG dictionary.
        device       : torch.device.

    Returns:
        model       : Model loaded from the best checkpoint.
        history     : Dict of training metrics per epoch.
    """

    # --- Compute class weights for imbalance correction ---
    class_weights = compute_class_weights(
        config["train_csv"], device
    )

    # --- Loss function: CrossEntropy with label smoothing ---
    # label_smoothing=0.1: instead of hard 1.0/0.0 targets,
    # uses 0.9/0.033 per class → prevents overconfident predictions
    criterion = nn.CrossEntropyLoss(
        weight=class_weights,
        label_smoothing=config["label_smoothing"]
    )

    # --- Optimiser: AdamW (Adam with decoupled weight decay) ---
    optimizer = optim.AdamW(
        model.parameters(),
        lr=config["learning_rate"],
        weight_decay=config["weight_decay"]
    )

    # --- LR Scheduler: ReduceLROnPlateau ---
    # Halves the LR when val macro F1 stops improving for 5 epochs
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",       # Maximise val F1 (not minimise)
        factor=0.5,       # Halve the LR when triggered
        patience=5,       # Wait 5 epochs before reducing
        min_lr=1e-7,      # Never go below this LR
        verbose=True,
    )

    # --- Early stopping variables ---
    best_val_f1       = 0.0
    epochs_no_improve = 0
    best_model_path   = os.path.join(config["save_dir"], config["cnn_filename"])

    # --- Training history storage ---
    history = {
        "train_loss": [], "train_acc": [],
        "val_loss": [],   "val_acc": [],  "val_f1": [],
    }

    print(f"\n{'='*60}")
    print(f"TRAINING: RespiratoryNet CNN")
    print(f"  Total parameters: {model.count_parameters():,}")
    print(f"  Device: {device}")
    print(f"{'='*60}")

    # --- Epoch loop ---
    for epoch in range(1, config["num_epochs"] + 1):
        epoch_start = time.time()

        # Training pass
        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, criterion, device
        )

        # Validation pass
        val_loss, val_acc, val_f1, _, _ = validate_one_epoch(
            model, val_loader, criterion, device
        )

        # Step the LR scheduler based on validation F1
        scheduler.step(val_f1)

        # Record history
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["val_f1"].append(val_f1)

        epoch_time = time.time() - epoch_start
        current_lr = optimizer.param_groups[0]["lr"]

        print(
            f"Epoch [{epoch:3d}/{config['num_epochs']}] "
            f"| TrainLoss: {train_loss:.4f} TrainAcc: {train_acc:.4f} "
            f"| ValLoss: {val_loss:.4f} ValAcc: {val_acc:.4f} ValF1: {val_f1:.4f} "
            f"| LR: {current_lr:.2e} | {epoch_time:.1f}s"
        )

        # --- Checkpoint: save if best val F1 ---
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            epochs_no_improve = 0
            # Save model state dict (weights only, not the whole model object)
            torch.save(model.state_dict(), best_model_path)
            print(f"  ↑ New best Val F1: {best_val_f1:.4f} — model saved.")
        else:
            epochs_no_improve += 1

        # --- Early stopping check ---
        if epochs_no_improve >= config["early_stop_patience"]:
            print(f"\n[EARLY STOP] No improvement for {config['early_stop_patience']} epochs.")
            print(f"[EARLY STOP] Best Val F1: {best_val_f1:.4f}")
            break

    # --- Reload best checkpoint ---
    print(f"\n[TRAIN] Loading best model from: {best_model_path}")
    model.load_state_dict(torch.load(best_model_path, map_location=device))

    return model, history

=============================================================================
SECTION 11: PLOT TRAINING CURVES
=============================================================================

In [34]:
def plot_training_curves(history, save_dir, save=True):
    """
    Plots training and validation loss, accuracy, and F1 curves over epochs.

    Args:
        history  (dict): Output from train_cnn() containing per-epoch metrics.
        save_dir (str) : Directory to save the plot.
        save     (bool): Whether to save to disk.
    """

    epochs = range(1, len(history["train_loss"]) + 1)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # Loss curve
    axes[0].plot(epochs, history["train_loss"], label="Train Loss", color="steelblue")
    axes[0].plot(epochs, history["val_loss"],   label="Val Loss",   color="orangered")
    axes[0].set_title("Loss over Epochs")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    # Accuracy curve
    axes[1].plot(epochs, history["train_acc"], label="Train Acc", color="steelblue")
    axes[1].plot(epochs, history["val_acc"],   label="Val Acc",   color="orangered")
    axes[1].set_title("Accuracy over Epochs")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
    axes[1].legend(); axes[1].grid(True, alpha=0.3)

    # Val F1 curve
    axes[2].plot(epochs, history["val_f1"], label="Val Macro F1", color="seagreen")
    axes[2].set_title("Validation Macro F1 (PRIMARY)")
    axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("Macro F1")
    axes[2].legend(); axes[2].grid(True, alpha=0.3)

    plt.suptitle("CNN Training History", fontsize=14, fontweight="bold")
    plt.tight_layout()

    if save:
        filepath = os.path.join(save_dir, "CNN_training_history.png")
        plt.savefig(filepath, dpi=150)
        print(f"[PLOT] Training curves saved to: {filepath}")

    plt.show()

=============================================================================
SECTION 12: EVALUATE CNN ON A DATALOADER
=============================================================================

In [35]:
def evaluate_cnn(model, loader, device, split_name="Test"):
    """
    Full evaluation of the CNN on a DataLoader split.

    Reports: Accuracy, Precision, Recall, F1 (macro + per class), Confusion Matrix.

    Args:
        model      : Trained RespiratoryNet (in eval mode).
        loader     : DataLoader (val or test).
        device     : torch.device.
        split_name : Label for the split being evaluated.

    Returns:
        metrics (dict): All computed metrics.
        all_preds  (list): Predicted class indices.
        all_labels (list): True class indices.
    """

    print(f"\n{'='*60}")
    print(f"CNN EVALUATION — {split_name} Set")
    print(f"{'='*60}")

    model.eval()
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for specs, labels in loader:
            specs  = specs.to(device)
            logits = model(specs)
            preds  = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    # Map indices back to class names for reporting
    target_names = [IDX_TO_LABEL[i] for i in sorted(IDX_TO_LABEL.keys())]

    # Metrics
    acc      = accuracy_score(all_labels, all_preds)
    prec_mac = precision_score(all_labels, all_preds, average="macro",  zero_division=0)
    rec_mac  = recall_score(all_labels,   all_preds, average="macro",  zero_division=0)
    f1_mac   = f1_score(all_labels,       all_preds, average="macro",  zero_division=0)
    prec_per = precision_score(all_labels, all_preds, average=None, zero_division=0)
    rec_per  = recall_score(all_labels,   all_preds, average=None, zero_division=0)
    f1_per   = f1_score(all_labels,       all_preds, average=None, zero_division=0)
    cm       = confusion_matrix(all_labels, all_preds)

    print(f"\n  Accuracy          : {acc:.4f}")
    print(f"\n  --- Macro Averages (PRIMARY METRICS) ---")
    print(f"  Precision (macro) : {prec_mac:.4f}")
    print(f"  Recall (macro)    : {rec_mac:.4f}  *** PRIMARY ***")
    print(f"  F1 (macro)        : {f1_mac:.4f}   *** PRIMARY ***")
    print(f"\n  --- Per-Class Breakdown ---")
    print(f"  {'Class':<10} {'Precision':>10} {'Recall':>10} {'F1':>10}")
    print(f"  {'-'*42}")
    for i, name in enumerate(target_names):
        print(f"  {name.capitalize():<10} {prec_per[i]:>10.4f} "
              f"{rec_per[i]:>10.4f} {f1_per[i]:>10.4f}")

    print(f"\n  --- Full Classification Report ---")
    print(classification_report(
        all_labels, all_preds,
        target_names=[n.capitalize() for n in target_names],
        zero_division=0
    ))

    print(f"  Confusion Matrix (rows=True, cols=Predicted):")
    print(f"  {cm}")

    metrics = {
        "accuracy": acc, "precision_macro": prec_mac,
        "recall_macro": rec_mac, "f1_macro": f1_mac,
        "precision_per_class": prec_per, "recall_per_class": rec_per,
        "f1_per_class": f1_per, "confusion_matrix": cm,
    }
    return metrics, all_preds, all_labels

In [36]:
def plot_cnn_confusion_matrix(cm, split_name, save_dir, label_names, save=True):
    """
    Plots the CNN confusion matrix for the 3-class problem.

    Args:
        cm         : Confusion matrix from sklearn.
        split_name : "Validation" or "Test"
        save_dir   : Directory to save the PNG.
        label_names: List of class name strings in order.
        save       : Whether to save to disk.
    """

    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Greens",
        xticklabels=[n.capitalize() for n in label_names],
        yticklabels=[n.capitalize() for n in label_names],
        ax=ax, linewidths=0.5, linecolor="gray",
    )
    ax.set_xlabel("Predicted", fontsize=12)
    ax.set_ylabel("True", fontsize=12)
    ax.set_title(f"CNN — {split_name} Confusion Matrix", fontsize=13)
    plt.tight_layout()

    if save:
        filepath = os.path.join(save_dir, f"CNN_{split_name}_confusion_matrix.png".replace(" ", "_"))
        plt.savefig(filepath, dpi=150)
        print(f"[PLOT] Confusion matrix saved to: {filepath}")

    plt.show()

=============================================================================
SECTION 13: RUN THE FULL TRAINING PIPELINE
=============================================================================

In [45]:
# --- Create DataLoaders ---
train_loader, val_loader, test_loader = create_dataloaders(
    config=CONFIG,
    transform=MEL_TRANSFORM,
)

[DATASET] Loaded 6697 samples from C:\Users\Admin\OneDrive\Desktop\. respiratory\1 final datasets, data info file, data distributions, final models\CNN models\only_paths_train_vowel_and_cough_ros_healthy_dropped.csv
  Asthma: 2443
  COPD: 2149
  COVID: 2105
[DATASET] Loaded 940 samples from C:\Users\Admin\OneDrive\Desktop\. respiratory\1 final datasets, data info file, data distributions, final models\CNN models\only_paths_val_vowel_and_cough_healthy_dropped.csv
  Asthma: 470
  COPD: 262
  COVID: 208
[DATASET] Loaded 786 samples from C:\Users\Admin\OneDrive\Desktop\. respiratory\1 final datasets, data info file, data distributions, final models\CNN models\only_paths_test_vowel_and_cough_healthy_dropped.csv
  Asthma: 346
  COPD: 245
  COVID: 195

[LOADER] Train batches: 209
[LOADER] Val batches  : 30
[LOADER] Test batches : 25


In [46]:
# --- Build the CNN model ---
cnn_model = RespiratoryNet(
    num_classes=CONFIG["num_classes"],
    n_mels=CONFIG["n_mels"],
).to(DEVICE)

In [47]:
print(f"\n[MODEL] Total trainable parameters: {cnn_model.count_parameters():,}")


[MODEL] Total trainable parameters: 421,859


In [52]:
# --- Train ---
cnn_model, history = train_cnn(cnn_model, train_loader, val_loader, CONFIG, DEVICE)


[CLASS WEIGHTS] Class counts: [2443. 2149. 2105.]
[CLASS WEIGHTS] Weights: tensor([0.9098, 1.0343, 1.0559])


TypeError: ReduceLROnPlateau.__init__() got an unexpected keyword argument 'verbose'

In [ ]:
# --- Plot training curves ---
plot_training_curves(history, CONFIG["save_dir"], save=CONFIG["save_plots"])

In [ ]:
# --- Evaluate on validation set ---
val_metrics, _, _ = evaluate_cnn(cnn_model, val_loader, DEVICE, split_name="Validation")

In [ ]:
if CONFIG["save_plots"]:
    plot_cnn_confusion_matrix(
        val_metrics["confusion_matrix"], "Validation",
        CONFIG["save_dir"], list(LABEL_MAP.keys())
    )

In [ ]:
# --- Evaluate on test set ---
test_metrics, _, _ = evaluate_cnn(cnn_model, test_loader, DEVICE, split_name="Test")

In [ ]:
if CONFIG["save_plots"]:
    plot_cnn_confusion_matrix(
        test_metrics["confusion_matrix"], "Test",
        CONFIG["save_dir"], list(LABEL_MAP.keys())
    )

=============================================================================
SECTION 14: SAVE CNN MODEL (already saved as best checkpoint)
=============================================================================
The best model was already saved during training (best val F1 checkpoint).
Path: CONFIG["save_dir"] / CONFIG["cnn_filename"]
It's saved as a state_dict (weights only), not the full model object.

In [ ]:
print(f"\n[SAVE] CNN best model is already saved at:")
print(f"       {os.path.join(CONFIG['save_dir'], CONFIG['cnn_filename'])}")

To reload the CNN later:
model = RespiratoryNet(num_classes=3, n_mels=128)
model.load_state_dict(torch.load('layer2_cnn.pth', map_location=device))
model.eval()

=============================================================================
SECTION 15: COMBINED TWO-LAYER PIPELINE
=============================================================================
This section defines the TwoLayerRespiratoryClassifier that stacks
Layer 1 (HistGradBoost or XGBoost) and Layer 2 (CNN) into one object.

Inference flow:
  Input: (feature_row [120 values], cough_wav_path, vowel_wav_path)
    Step 1 → Layer 1 predicts: healthy (0) or diseased (1)
    If healthy → return "healthy" immediately
    If diseased → Step 2: Layer 2 predicts: asthma / copd / covid
    Return final disease label
=============================================================================

In [ ]:
class TwoLayerRespiratoryClassifier:
    """
    Full two-layer inference pipeline for respiratory disease classification.

    Layer 1: Trained HistGradBoost or XGBoost model (sklearn)
             Input: 120-feature CSV row
             Output: 0 (healthy) or 1 (diseased)

    Layer 2: Trained RespiratoryNet CNN (PyTorch)
             Input: cough WAV path + vowel WAV path
             Output: asthma / copd / covid

    Args:
        layer1_model     : Fitted sklearn model (loaded via joblib or xgboost).
        layer2_model     : Trained RespiratoryNet (nn.Module), already in eval mode.
        mel_transform    : Mel spectrogram transform (nn.Sequential).
        label_map        (dict): {'asthma':0, 'copd':1, 'covid':2}
        device           : torch.device for CNN inference.
        sample_rate      (int): Audio sample rate.
        clip_duration    (float): Fixed audio clip duration in seconds.
        layer1_threshold (float): Probability threshold for diseased classification.
                                  Lower = more sensitive (catches more disease cases).
                                  Default 0.5; lower to 0.4 for higher sensitivity.
    """

    def __init__(
        self,
        layer1_model,
        layer2_model,
        mel_transform,
        label_map,
        device,
        sample_rate=16000,
        clip_duration=5.0,
        layer1_threshold=0.5,
    ):
        self.layer1_model      = layer1_model
        self.layer2_model      = layer2_model
        self.mel_transform     = mel_transform
        self.idx_to_label      = {v: k for k, v in label_map.items()}
        self.device            = device
        self.sample_rate       = sample_rate
        self.clip_duration     = clip_duration
        self.layer1_threshold  = layer1_threshold

        # Put the CNN in evaluation mode (disables Dropout, freezes BatchNorm)
        self.layer2_model.eval()
        print("[PIPELINE] TwoLayerRespiratoryClassifier initialised.")

    def predict_single(self, feature_row, cough_wav_path, vowel_wav_path):
        """
        Predicts the health status of a single patient.

        Args:
            feature_row      (array-like): 1D array/list of 120 CSV features.
                                           Must NOT include the ID column.
            cough_wav_path   (str)       : Path to the cough WAV file.
            vowel_wav_path   (str)       : Path to the vowel WAV file.

        Returns:
            result (dict): {
                'layer1_label'  : 'healthy' or 'diseased',
                'layer1_prob'   : Probability of being diseased (float),
                'layer2_label'  : 'asthma'/'copd'/'covid'/None (None if healthy),
                'final_label'   : Final predicted label (string),
            }
        """

        # ---- LAYER 1: Healthy vs Diseased ----
        # Reshape to (1, n_features) for sklearn's expected input shape
        X = np.array(feature_row).reshape(1, -1)

        # Get disease probability (probability of class 1 = diseased)
        diseased_prob = self.layer1_model.predict_proba(X)[0, 1]

        # Apply threshold (default 0.5, lower for higher recall on disease)
        is_diseased = diseased_prob >= self.layer1_threshold

        if not is_diseased:
            # Patient is healthy — skip Layer 2
            return {
                "layer1_label" : "healthy",
                "layer1_prob"  : float(diseased_prob),
                "layer2_label" : None,
                "final_label"  : "healthy",
            }

        # ---- LAYER 2: Disease-type classification ----
        # Load and process both audio files
        cough_spec = load_and_preprocess_audio(
            cough_wav_path, self.sample_rate,
            self.clip_duration, self.mel_transform
        )
        vowel_spec = load_and_preprocess_audio(
            vowel_wav_path, self.sample_rate,
            self.clip_duration, self.mel_transform
        )

        # Concatenate along time axis (same as training)
        combined = torch.cat([cough_spec, vowel_spec], dim=2)

        # Add batch dimension: (1, 1, n_mels, 2T)
        combined = combined.unsqueeze(0).to(self.device)

        # CNN forward pass (no gradient needed)
        with torch.no_grad():
            logits     = self.layer2_model(combined)        # (1, num_classes)
            probs      = torch.softmax(logits, dim=1)       # Convert to probabilities
            pred_idx   = torch.argmax(probs, dim=1).item()  # Winning class index

        disease_label = self.idx_to_label[pred_idx]

        return {
            "layer1_label" : "diseased",
            "layer1_prob"  : float(diseased_prob),
            "layer2_label" : disease_label,
            "final_label"  : disease_label,
        }

    def predict_batch_from_df(self, features_df, audio_df):
        """
        Runs the full pipeline on a batch of patients.

        Args:
            features_df (pd.DataFrame): Feature matrix (N × 120), no ID column.
            audio_df    (pd.DataFrame): DataFrame with columns:
                                        ['id', 'cough_path', 'vowel_path']
                                        Index must align with features_df.

        Returns:
            results (list of dict): One result dict per patient.
        """

        results = []
        for i in range(len(features_df)):
            feature_row    = features_df.iloc[i].values
            cough_path     = audio_df.iloc[i]["cough_path"]
            vowel_path     = audio_df.iloc[i]["vowel_path"]
            result         = self.predict_single(feature_row, cough_path, vowel_path)
            results.append(result)

        return results

=============================================================================
SECTION 16: LOAD LAYER 1 MODEL AND ASSEMBLE THE PIPELINE
=============================================================================

In [ ]:
def load_layer1_model(layer1_path, layer1_type):
    """
    Loads the saved Layer 1 model from disk.

    Args:
        layer1_path (str): Path to the saved model file.
        layer1_type (str): 'histgradboost' or 'xgboost'

    Returns:
        Loaded sklearn-compatible model.
    """

    if layer1_type == "histgradboost":
        # HistGradBoost was saved with joblib
        model = joblib.load(layer1_path)
        print(f"[LOAD] HistGradBoost loaded from: {layer1_path}")

    elif layer1_type == "xgboost":
        # XGBoost needs to be loaded into an XGBClassifier instance
        from xgboost import XGBClassifier
        model = XGBClassifier()
        model.load_model(layer1_path)
        print(f"[LOAD] XGBoost loaded from: {layer1_path}")

    else:
        raise ValueError(f"Unknown layer1_type: '{layer1_type}'. Use 'histgradboost' or 'xgboost'.")

    return model

In [ ]:
def load_layer2_model(cnn_path, num_classes, n_mels, device):
    """
    Loads the saved CNN state dict into a RespiratoryNet model.

    Args:
        cnn_path    (str): Path to the .pth state dict file.
        num_classes (int): Number of disease classes (3).
        n_mels      (int): Number of mel bins (128).
        device      : torch.device.

    Returns:
        Loaded RespiratoryNet in eval mode.
    """

    model = RespiratoryNet(num_classes=num_classes, n_mels=n_mels)
    model.load_state_dict(torch.load(cnn_path, map_location=device))
    model.to(device)
    model.eval()
    print(f"[LOAD] CNN loaded from: {cnn_path}")
    return model

In [ ]:
# --- Load both saved models ---
layer1 = load_layer1_model(CONFIG["layer1_path"], CONFIG["layer1_type"])

In [ ]:
layer2 = load_layer2_model(
    cnn_path=os.path.join(CONFIG["save_dir"], CONFIG["cnn_filename"]),
    num_classes=CONFIG["num_classes"],
    n_mels=CONFIG["n_mels"],
    device=DEVICE,
)

In [ ]:
# --- Assemble the combined pipeline ---
pipeline = TwoLayerRespiratoryClassifier(
    layer1_model=layer1,
    layer2_model=layer2,
    mel_transform=MEL_TRANSFORM,
    label_map=LABEL_MAP,
    device=DEVICE,
    sample_rate=CONFIG["sample_rate"],
    clip_duration=CONFIG["clip_duration"],
    layer1_threshold=0.5,         # Adjust lower (e.g., 0.4) to improve disease recall
)

=============================================================================
SECTION 17: SAVE THE COMBINED PIPELINE
=============================================================================

In [ ]:
def save_pipeline(pipeline, save_dir, filename):
    """
    Saves the TwoLayerRespiratoryClassifier object to disk using pickle.

    IMPORTANT: The CNN state_dict is part of the pipeline object.
               The sklearn Layer 1 model is also embedded.
               Both are serialised together into one file.

    NOTE: For large models or cross-platform sharing, it's safer to save
          Layer 1 and Layer 2 separately (already done above), and only
          use this combined pickle for local deployment.

    Args:
        pipeline : Fitted TwoLayerRespiratoryClassifier instance.
        save_dir : Directory to save the pipeline.
        filename : Filename (e.g., 'full_pipeline.pkl').

    Returns:
        save_path (str): Full path to the saved pipeline.
    """

    save_path = os.path.join(save_dir, filename)
    with open(save_path, "wb") as f:
        pickle.dump(pipeline, f)
    print(f"\n[SAVE] Full pipeline saved to: {save_path}")
    return save_path

In [ ]:
pipeline_path = save_pipeline(pipeline, CONFIG["save_dir"], CONFIG["pipeline_filename"])

=============================================================================
SECTION 18: RELOAD AND VERIFY THE COMBINED PIPELINE
=============================================================================

In [ ]:
def load_pipeline(pipeline_path):
    """
    Reloads a saved TwoLayerRespiratoryClassifier from disk.

    Args:
        pipeline_path (str): Full path to the .pkl file.

    Returns:
        Loaded TwoLayerRespiratoryClassifier instance.
    """

    with open(pipeline_path, "rb") as f:
        loaded_pipeline = pickle.load(f)
    print(f"[LOAD] Pipeline reloaded from: {pipeline_path}")
    return loaded_pipeline

In [ ]:
# --- Reload the pipeline ---
loaded_pipeline = load_pipeline(pipeline_path)

In [ ]:
# --- Quick sanity check with a dummy prediction ---
print("\n[VERIFY] Running a dummy prediction to verify pipeline integrity...")
print("[VERIFY] NOTE: Replace the paths below with real files for actual verification.")

Uncomment and fill in real paths to run the actual test:
test_result = loaded_pipeline.predict_single(
    feature_row  = X_test.iloc[0].values,         # First test row features
    cough_wav_path = "/content/audio/cough_001.wav",
    vowel_wav_path = "/content/audio/vowel_001.wav",
)
print(f"[VERIFY] Prediction result: {test_result}")

In [ ]:
print("[VERIFY] Pipeline reload verified. Replace dummy paths to test with real audio.")

=============================================================================
SECTION 19: FINAL SUMMARY
=============================================================================

In [ ]:
print("\n" + "="*60)
print("FINAL SUMMARY — Full Two-Layer Respiratory Classifier")
print("="*60)
print(f"\nLayer 2 CNN — Validation:")
print(f"  Accuracy  : {val_metrics['accuracy']:.4f}")
print(f"  Recall M  : {val_metrics['recall_macro']:.4f}  *** PRIMARY ***")
print(f"  F1 Macro  : {val_metrics['f1_macro']:.4f}   *** PRIMARY ***")
print(f"\nLayer 2 CNN — Test:")
print(f"  Accuracy  : {test_metrics['accuracy']:.4f}")
print(f"  Recall M  : {test_metrics['recall_macro']:.4f}  *** PRIMARY ***")
print(f"  F1 Macro  : {test_metrics['f1_macro']:.4f}   *** PRIMARY ***")
print(f"\nSaved files:")
print(f"  Layer 1 model  : {CONFIG['layer1_path']}")
print(f"  Layer 2 CNN    : {os.path.join(CONFIG['save_dir'], CONFIG['cnn_filename'])}")
print(f"  Full pipeline  : {pipeline_path}")
print("="*60)
print("\n[DONE] All models trained, evaluated, saved, and verified.")

=============================================================================
END OF FILE: layer2_cnn.py
=============================================================================